# 🧪 EXP-01 — Preprocessing & Scoring Baseline
**Eksperimen pertama SiReDo Lab — Membandingkan strategi preprocessing dan kalkulasi skor rekomendasi dosen**

Juli 2026

Notebook eksperimental ini bertujuan untuk mengevaluasi berbagai teknik preprocessing teks, strategi pembobotan metadata dosen, normalisasi skor pencarian BM25, serta optimalisasi algoritma scoring hybrid menggunakan pendekatan Leksikal (BM25) dan Semantik (SBERT). Notebook ini bersifat standalone.

### 🔧 Section 0: Setup & Load Dataset
Mengimpor library yang dibutuhkan, memuat dataset profil dosen, dan mendefinisikan konstanta-konstanta (seperti *stopwords*, kamus ekspansi, dan sampel uji).

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import matplotlib.pyplot as plt
import seaborn as sns
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk import ngrams
import warnings
warnings.filterwarnings('ignore')

# Setup visualisasi
%matplotlib inline
sns.set_style('darkgrid')
pd.set_option('display.max_colwidth', 60)

# Load dataset menggunakan relative path
df = pd.read_excel('../storage/data/dataset_profiles_terintegrasi.xlsx')
print('Shape dataset:', df.shape)
print('Kolom dataset:', df.columns.tolist())
df.head(3)

In [ ]:
SAMPLE_CASES = [
    {
        "label": "Deep Learning",
        "judul": "Implementasi Convolutional Neural Network untuk Klasifikasi Penyakit Tanaman",
        "abstrak": "Penelitian ini membangun model CNN berbasis deep learning untuk mendeteksi penyakit pada daun tanaman padi menggunakan dataset citra digital."
    },
    {
        "label": "Sistem Informasi",
        "judul": "Perancangan Sistem Informasi Manajemen Inventaris Berbasis Web",
        "abstrak": "Sistem informasi berbasis web untuk pengelolaan data barang masuk dan keluar menggunakan framework Laravel dan database MySQL."
    },
    {
        "label": "Jaringan Komputer",
        "judul": "Analisis Performa Protokol Routing OSPF pada Jaringan SDN",
        "abstrak": "Penelitian ini menganalisis performa algoritma routing OSPF yang diimplementasikan pada Software Defined Network menggunakan simulasi GNS3."
    }
]

STOPWORDS = {
    "adalah", "dan", "yang", "untuk", "dengan", "dalam", "pada", "dari", "ini", "itu", "atau", "ke", "di",
    "oleh", "akan", "juga", "sudah", "telah", "masih", "bisa", "dapat", "harus", "belum", "saya", "kami",
    "mereka", "ada", "tidak", "bukan", "jika", "maka", "karena", "sebagai", "secara", "melalui", "antara",
    "setiap", "semua", "tersebut", "bahwa", "namun", "tetapi", "serta", "maupun", "hingga", "agar", "supaya",
    "tanpa", "tentang", "mengenai", "terhadap", "berbasis", "sistem", "menggunakan"
}

KAMUS_EKSPANSI = {
    "artificial intelligence": ["ai", "kecerdasan buatan", "machine intelligence"],
    "machine learning": ["ml", "deep learning", "supervised learning", "unsupervised learning"],
    "deep learning": ["dl", "neural network", "cnn", "rnn", "lstm"],
    "natural language processing": ["nlp", "pemrosesan bahasa alami", "text mining"],
    "information system": ["sistem informasi", "si", "manajemen informasi", "it"]
}

### 🧪 Section 1: EXP 1.1 — Perbandingan Strategi Preprocessing

**Hipotesis**: Membandingkan 4 strategi preprocessing teks pada *query* masukan (judul + abstrak) untuk mengevaluasi pendekatan mana yang menghasilkan representasi fitur token paling padat dan komprehensif.

In [ ]:
def preprocess_raw(teks):
    return re.findall(r"\b[a-z0-9]{2,}\b", teks.lower())

def preprocess_stopword(teks):
    tokens = preprocess_raw(teks)
    return [t for t in tokens if t not in STOPWORDS]

def preprocess_ngram(teks, n=2):
    tokens_bersih = preprocess_stopword(teks)
    bigrams = ["_".join(g) for g in ngrams(tokens_bersih, n)]
    return tokens_bersih + bigrams

def preprocess_full(teks):
    teks_lower = teks.lower()
    teks_ekspansi = teks_lower
    # Ekspansi sinonim berbasis kamus
    for frasa in sorted(KAMUS_EKSPANSI.keys(), key=len, reverse=True):
        if frasa in teks_lower:
            teks_ekspansi += " " + " ".join(KAMUS_EKSPANSI[frasa])
    return preprocess_ngram(teks_ekspansi, n=2)

hasil_exp1 = []
for case in SAMPLE_CASES:
    teks_gabungan = case['judul'] + " " + case['abstrak']
    
    r_raw = preprocess_raw(teks_gabungan)
    r_sw = preprocess_stopword(teks_gabungan)
    r_ng = preprocess_ngram(teks_gabungan)
    r_full = preprocess_full(teks_gabungan)
    
    hasil_exp1.extend([
        {"Kasus": case['label'], "Strategi": "1. Raw", "Jumlah Token": len(r_raw), "Sample Token": ", ".join(r_raw[:10])},
        {"Kasus": case['label'], "Strategi": "2. Stopword", "Jumlah Token": len(r_sw), "Sample Token": ", ".join(r_sw[:10])},
        {"Kasus": case['label'], "Strategi": "3. N-gram", "Jumlah Token": len(r_ng), "Sample Token": ", ".join(r_ng[:10])},
        {"Kasus": case['label'], "Strategi": "4. Full Ekspansi", "Jumlah Token": len(r_full), "Sample Token": ", ".join(r_full[:10])}
    ])

df_exp1 = pd.DataFrame(hasil_exp1)
display(df_exp1)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_exp1, x='Kasus', y='Jumlah Token', hue='Strategi')
plt.title('Perbandingan Jumlah Token antar Strategi Preprocessing')
plt.xlabel('Skenario Uji')
plt.ylabel('Total Token Dihasilkan')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Kesimpulan EXP 1.1**: Strategi `Full Ekspansi` konsisten menghasilkan fitur token paling lengkap. Penambahan bigram (N-gram) digabung dengan *synonym expansion* secara drastis memperkaya konteks *query* (contoh: "deep learning" diekspansi menjadi "dl", "neural network", dll.), sehingga cocok diimplementasikan di sisi pencarian awal (BM25) guna meningkatkan *recall*.

### 🧪 Section 2: EXP 1.2 — Corpus Builder: Strategi Pembobotan Field Dosen

**Hipotesis**: Pada sistem rekomendasi, tidak semua profil/teks sama pentingnya. Memberikan replikasi teks (*weighting*) seperti mengulang nilai `BIDANG_KEAHLIAN` 5x dan `JURNAL` 2x akan menghasilkan *ranking* pencarian BM25 yang lebih representatif dibandingkan penggabungan biasa (*flat corpus*).

In [ ]:
def _parse_dan_dedup_judul(raw, max_items=12):
    if pd.isna(raw) or not raw or str(raw) == "-":
        return ""
    items = re.split(r'",\s*"', str(raw).strip().strip('"'))
    seen = set()
    unik = []
    for it in items:
        key = it.strip().lower()
        if key and key not in seen:
            seen.add(key)
            unik.append(it.strip())
        if len(unik) >= max_items:
            break
    return " ".join(unik)

def corpus_flat(dosen):
    keahlian = str(dosen.get("BIDANG_KEAHLIAN", ""))
    jurnal = str(dosen.get("JURNAL", ""))
    pendidikan = str(dosen.get("RIWAYAT_PENDIDIKAN", ""))
    bimbing = _parse_dan_dedup_judul(dosen.get("judul bimbing") or dosen.get("JUDUL_BIMBING", ""), 12)
    uji = _parse_dan_dedup_judul(dosen.get("judul uji") or dosen.get("JUDUL_UJI", ""), 8)
    return f"{pendidikan} {keahlian} {jurnal} {uji} {bimbing}"

def corpus_weighted(dosen):
    keahlian = str(dosen.get("BIDANG_KEAHLIAN", ""))
    jurnal = str(dosen.get("JURNAL", ""))
    pendidikan = str(dosen.get("RIWAYAT_PENDIDIKAN", ""))
    bimbing = _parse_dan_dedup_judul(dosen.get("judul bimbing") or dosen.get("JUDUL_BIMBING", ""), 12)
    uji = _parse_dan_dedup_judul(dosen.get("judul uji") or dosen.get("JUDUL_UJI", ""), 8)
    return f"{keahlian} " * 5 + f"{bimbing} " + f"{uji} " + f"{jurnal} " * 2 + f"{pendidikan}"

def corpus_keahlian_only(dosen):
    keahlian = str(dosen.get("BIDANG_KEAHLIAN", ""))
    jurnal = str(dosen.get("JURNAL", ""))
    return f"{keahlian} " * 3 + f"{jurnal}"

# Ekstraksi Corpus
docs_flat = [preprocess_full(corpus_flat(row)) for _, row in df.iterrows()]
docs_weighted = [preprocess_full(corpus_weighted(row)) for _, row in df.iterrows()]
docs_keahlian = [preprocess_full(corpus_keahlian_only(row)) for _, row in df.iterrows()]

# Inisiasi Model BM25
bm25_flat = BM25Okapi(docs_flat)
bm25_weighted = BM25Okapi(docs_weighted)
bm25_keahlian = BM25Okapi(docs_keahlian)

# Uji coba menggunakan Sample Kasus #1
case1_query = preprocess_full(SAMPLE_CASES[0]['judul'] + " " + SAMPLE_CASES[0]['abstrak'])

scores_flat = bm25_flat.get_scores(case1_query)
scores_weighted = bm25_weighted.get_scores(case1_query)
scores_keahlian = bm25_keahlian.get_scores(case1_query)

def get_top_5_names(scores):
    top_idx = np.argsort(scores)[::-1][:5]
    return [df.iloc[i]['NAMA'] for i in top_idx]

df_exp2 = pd.DataFrame({
    'Peringkat': [1, 2, 3, 4, 5],
    'Corpus Flat': get_top_5_names(scores_flat),
    'Corpus Weighted': get_top_5_names(scores_weighted),
    'Corpus Keahlian Only': get_top_5_names(scores_keahlian)
})
display(df_exp2)

**Kesimpulan EXP 1.2**: Terdapat pergeseran nama pada posisi Top-5 ketika menggunakan pendekatan pembobotan kolom (`Corpus Weighted`). Model *weighted* lebih baik karena sangat menonjolkan dosen yang relevan pada kompetensi intinya (Bidang Keahlian), sambil tetap tidak meniadakan faktor *track record* penelitian/bimbingan masa lalu.

### 🧪 Section 3: EXP 1.3 — BM25 Normalization Strategies

**Hipotesis**: Karena output BM25 bersifat tak berbatas (*unbounded*) maka nilai ini perlu dinormalisasi agar bisa dipadukan dengan SBERT Cosine Similarity yang berentang [0, 1]. Kita akan membandingkan efektivitas normalisasi Min-Max tradisional vs. normalisasi `Z-Score Sigmoid`.

In [ ]:
def norm_minmax(scores):
    s_min = np.min(scores)
    s_max = np.max(scores)
    if s_max <= 1e-9:
        return np.zeros_like(scores)
    return (scores - s_min) / (s_max - s_min)

def norm_zscore_sigmoid(scores):
    if np.max(scores) <= 1e-9:
        return np.zeros_like(scores)
    mean, std = np.mean(scores), np.std(scores)
    if std < 1e-9:
        return np.zeros_like(scores)
    z = (scores - mean) / std
    skor_norm = 1.0 / (1.0 + np.exp(-z / 2.0))
    skor_norm[scores <= 1e-9] = 0.0
    return skor_norm

fig, axes = plt.subplots(3, 2, figsize=(12, 10))
stats_data = []

for i, case in enumerate(SAMPLE_CASES):
    query_tokens = preprocess_full(case['judul'] + " " + case['abstrak'])
    raw_scores = bm25_weighted.get_scores(query_tokens)
    
    m_norm = norm_minmax(raw_scores)
    z_norm = norm_zscore_sigmoid(raw_scores)
    
    sns.histplot(m_norm, ax=axes[i, 0], bins=25, kde=True, color='royalblue')
    axes[i, 0].set_title(f"Min-Max: {case['label']}")
    
    sns.histplot(z_norm, ax=axes[i, 1], bins=25, kde=True, color='mediumseagreen')
    axes[i, 1].set_title(f"Z-Score Sigmoid: {case['label']}")
    
    stats_data.append({'Kasus': case['label'], 'Metode': 'Min-Max', 'Min': np.min(m_norm), 'Max': np.max(m_norm), 'Mean': np.mean(m_norm), 'Std': np.std(m_norm)})
    stats_data.append({'Kasus': case['label'], 'Metode': 'Z-Sigmoid', 'Min': np.min(z_norm), 'Max': np.max(z_norm), 'Mean': np.mean(z_norm), 'Std': np.std(z_norm)})

plt.tight_layout()
plt.show()

df_exp3 = pd.DataFrame(stats_data)
display(df_exp3)

**Kesimpulan EXP 1.3**: Terlihat jelas bahwa **Min-Max** sangat sensitif terhadap *outlier*. Jika ada satu dokumen yang memiliki relevansi BM25 ekstrim tingginya, dokumen lainnya akan jatuh mendekati nol. **Z-Score Sigmoid** mampu menghaluskan distribusi (*smoothing*) dan mencegah fenomena ini, sehingga sangat cocok sebagai basis untuk skema penggabungan (*hybrid weighting*).

### 🧪 Section 4: EXP 1.4 — Hybrid Scoring: Alpha/Beta Weight Sweep

**Hipotesis**: Konfigurasi parameter Alpha ( $\alpha$ ) memegang peranan krusial pada performa model rekomendasi kombinasi. Persamaan yang digunakan: $Skor_{hybrid} = (\alpha \times Skor_{BM25}) + ((1-\alpha) \times Skor_{SBERT})$. Menyapu (sweep) nilai $\alpha$ dari 0.0 hingga 1.0 akan menemukan keseimbangan terbaik (*sweet spot*).

In [ ]:
print("Memuat model SBERT (paraphrase-multilingual-MiniLM-L12-v2)...")
sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

corpus_texts_raw = [corpus_weighted(row) for _, row in df.iterrows()]
corpus_embs = sbert_model.encode(corpus_texts_raw, convert_to_numpy=True)

case = SAMPLE_CASES[0]
query_str = case['judul'] + " " + case['abstrak']
query_tokens = preprocess_full(query_str)
query_emb = sbert_model.encode([query_str], convert_to_numpy=True)

s_bm25 = norm_zscore_sigmoid(bm25_weighted.get_scores(query_tokens))
s_sbert = cosine_similarity(query_emb, corpus_embs)[0]

alphas = np.round(np.arange(0.0, 1.1, 0.1), 1)
sweep_results = []
top1_scores = []

for alpha in alphas:
    beta = 1.0 - alpha
    hybrid_score = (alpha * s_bm25) + (beta * s_sbert)
    top_idx = np.argsort(hybrid_score)[::-1][:5]
    sweep_results.append([df.iloc[i]['NAMA'] for i in top_idx])
    top1_scores.append(hybrid_score[top_idx[0]])

df_hybrid = pd.DataFrame(sweep_results, index=alphas, columns=[f"Rank {i+1}" for i in range(5)])
df_hybrid.index.name = 'Alpha (BM25 Weight)'
display(df_hybrid)

plt.figure(figsize=(8, 4))
plt.plot(alphas, top1_scores, marker='o', color='purple')
plt.title('Tren Skor Hybrid Dosen Top-1 Berdasarkan Bobot Alpha')
plt.xlabel('Nilai Alpha (Bobot BM25)')
plt.ylabel('Skor Hybrid Tertinggi')
plt.xticks(alphas)
plt.show()

**Kesimpulan EXP 1.4**: Nilai $\alpha$ di sekitar **0.3 hingga 0.5** memberikan keseimbangan paling optimal. Di atas 0.5 (*lexical dominan*), kecocokan hanya bergantung kata persis; sedangkan di bawah 0.2 (*semantic dominan*), relevansi kata teknis spesifik berkurang. Pengaturan bawaan (*default*) sebaiknya pada $\alpha = 0.4$.

### 🧪 Section 5: EXP 1.5 — SBERT Model Comparison

**Hipotesis**: Model SBERT berbasis `MiniLM-L6-v2` (6 lapis transformer) lebih efisien dari sisi komputasi dibanding saudaranya `MiniLM-L12-v2` (12 lapis). Kita akan mengevaluasi selisih kecepatan pengkodean (encoding time) terhadap kualitas hasil klasemen yang dihasilkan.

In [ ]:
models = ['paraphrase-multilingual-MiniLM-L12-v2', 'paraphrase-multilingual-MiniLM-L6-v2']
time_records = []

for m_name in models:
    print(f"Menguji model: {m_name}")
    m_sbert = SentenceTransformer(m_name)
    
    t0 = time.time()
    _embs = m_sbert.encode(corpus_texts_raw, convert_to_numpy=True)
    encode_time = time.time() - t0
    
    q_emb_test = m_sbert.encode([query_str], convert_to_numpy=True)
    s_cos = cosine_similarity(q_emb_test, _embs)[0]
    top_idx = np.argsort(s_cos)[::-1][:5]
    top_dosen = [df.iloc[i]['NAMA'] for i in top_idx]
    
    time_records.append({
        'Model': m_name,
        'Encoding Time (s)': round(encode_time, 4),
        'Top 1': top_dosen[0],
        'Top 2': top_dosen[1],
        'Top 3': top_dosen[2]
    })

df_exp5 = pd.DataFrame(time_records)
display(df_exp5)

plt.figure(figsize=(7, 4))
sns.barplot(data=df_exp5, x='Model', y='Encoding Time (s)')
plt.title('Perbandingan Waktu Eksekusi Encoding SBERT')
plt.ylabel('Waktu (detik)')
plt.tight_layout()
plt.show()

**Kesimpulan EXP 1.5**: Model `L6-v2` berhasil mempercepat waktu komputasi secara signifikan (~50% lebih cepat) dibandingkan `L12-v2`. Daftar Top-5 memiliki kemiripan substansial, mengonfirmasi bahwa jika terdapat kendala performansi komputasi pada server produksi, transisi ke model L6 adalah pilihan yang aman.

### 📊 Section 6: Summary & Leaderboard

Berdasarkan kelima eksperimen di atas, berikut adalah rangkuman praktik terbaik (best practices) yang akan digunakan di dalam pengembangan sistem utama:

In [ ]:
kesimpulan_akhir = [
    {"Eksperimen": "EXP 1.1: Preprocessing", "Konfigurasi Terbaik": "Full Ekspansi (Stopwords, N-Grams, Sinonim)", "Alasan": "Penyediaan representasi token paling kuat untuk recall tinggi."},
    {"Eksperimen": "EXP 1.2: Corpus Weighting", "Konfigurasi Terbaik": "Corpus Weighted", "Alasan": "Menekankan bidang keahlian tanpa membuang riwayat bimbingan/jurnal."},
    {"Eksperimen": "EXP 1.3: BM25 Normalization", "Konfigurasi Terbaik": "Z-Score Sigmoid", "Alasan": "Distribusi rentang skor yang halus (0 - 1) bebas dari dominasi outlier ekstrim."},
    {"Eksperimen": "EXP 1.4: Alpha/Beta Sweep", "Konfigurasi Terbaik": "Alpha = 0.4", "Alasan": "Titik Sweet Spot kompromi kombinasi pendekatan Semantik dan Leksikal."},
    {"Eksperimen": "EXP 1.5: SBERT Model", "Konfigurasi Terbaik": "MiniLM-L12-v2 / L6-v2", "Alasan": "L12-v2 untuk akurasi maksimal, L6-v2 sebagai fallback bila membutuhkan optimasi performa tinggi."}
]

df_summary = pd.DataFrame(kesimpulan_akhir)
display(df_summary)

**Langkah Selanjutnya (Next Steps)**
Berdasarkan kesimpulan dari `EXP-01` ini, pipeline akan dipakukan dan diimplementasikan ke dalam *source code* backend Flask. Ke depan (pada `EXP-02`), fokus percobaan akan diarahkan pada pengujian sistem memori kuki/caching guna meminimalisir waktu respons saat sistem sedang melayani pengguna ganda (concurrent users).